# FLUJO 1
- Simulo los datos
- Analizo la data
- Construyo la data que le pasare al modelo


# 1. Imports y rutas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import sys
from pathlib import Path

# ── Generadores de datos y contrato de artefactos (pipelines) ────────────────
from model_psbp_fd.pipelines import (
    ConfigEscenario1, generar_escenario_1, guardar_escenario,
    guardar_curvas, guardar_representacion, guardar_fpca,
    guardar_estandarizador, guardar_datasets_ar,
    guardar_hiperparametros, guardar_config_evaluacion,
    verificar_contrato,
)

# ── Preprocesamiento funcional (functions_models) ────────────────────────────
from model_psbp_fd.functions_models import (
    FunctionalRepresentation, FPCA_L2, base_en_grilla, DataStandardizer,
)

# ── Evaluacion: lineas base sobre el bloque de prueba (fit) ──────────────────
from model_psbp_fd.fit import tabla_baselines

# ── Utilidades ───────────────────────────────────────────────────────────────
from model_psbp_fd.utils import get_project_root

# ── Visualizacion ────────────────────────────────────────────────────────────
from model_psbp_fd.graphics import (
    plot_empirical_sample, plot_functional_mean, plot_functional_variance,
    plot_mean_and_variance, plot_fts_empirical, plot_fts_functional,
    plot_scatter_theta, plot_functional_comparison,
    plot_diagnostico_estandarizacion, plot_fpca_scree,plot_seleccion_basis,
    plot_rezagos_heatmap
)

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

## 1.1 Constantes a modificar según experimento 

In [ ]:
# Buscamos la raiz del proyecto
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT : {PROJECT_ROOT}")


# IDENTIFICAR EL EXPERIMENTO: variables globales y rutas de salida
BASENAME      = "modelo_experimento_1"
TT            = 1
SEED          = 41232
#TIMESTAMP     = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_ID = f"{BASENAME}_{TT}"#_{TIMESTAMP}"      # Nombre del experimento 
print(f"Experiment ID : {EXPERIMENT_ID}")
print(f"Seed (base)   : {SEED}")

## 1.2 Construccion de Rutas

In [ ]:
# Construccion de rutas 
_REPORT_DIR = PROJECT_ROOT / "reports" / "simulaciones" / EXPERIMENT_ID
_ARTEFACT_DIR = PROJECT_ROOT / "artefact" / "simulaciones" / EXPERIMENT_ID

PATHS = {
    "raw":          PROJECT_ROOT / "data" / "simulaciones" / "raw" / EXPERIMENT_ID,                         # Datos: raw + estandarizados
    "functional":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "functional" / EXPERIMENT_ID,    # Coeficientes funcionales + FPCA
    "predict":      PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict" / EXPERIMENT_ID,       # Datos predichos
    "out_report":   _REPORT_DIR,          # Figuras + métricas/config JSON
    "out_artefact": _ARTEFACT_DIR,        # Artefactos (serializados, por experimento)
}
for name, path in PATHS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"  {name:10s} → {path}")

# 2. Simulación

In [ ]:
# CONSTANTES DE LA SIMULACION — Escenario 1 (Algoritmo 1 del anexo)

def media_senoidal(tau):
    """mu(tau) = 5 + 2 sin(2 pi tau). Definida con nombre para que quede
    registrada de forma legible en simulation_config.json."""
    return 5.0 + 2.0 * np.sin(2.0 * np.pi * tau)

SIM_CFG = ConfigEscenario1(
    # -- Esquema de observacion (comun a todos los escenarios) --
    L         = 50,       # puntos de la grilla regular
    T         = 150,      # curvas retenidas
    burn_in   = 200,      # calentamiento descartado
    sigma_obs = 0.5,      # desv. estandar del ruido de medicion
    R         = 1,        # replicas (flujo 1: una; estudio Monte Carlo: 50)
    seed      = SEED,
    media_fn  = media_senoidal,
    # -- Operador autorregresivo --
    gamma     = 0.3,      # alcance del nucleo gaussiano
    hs_norm   = 0.7,      # ||Psi||_HS objetivo; < 1 garantiza estacionariedad
    # -- Innovacion funcional --
    sigma_eps = 1.0,      # escala de la innovacion
    ell       = 0.2,      # longitud de correlacion (suavidad de trayectorias)
)

for k, v in SIM_CFG.to_dict().items():
    print(f"  {k:<12}: {v}")


## 2.1 Simulación — Escenario 1: FAR(1) lineal gaussiano (Algoritmo 1 del anexo)


In [ ]:
# ── Generación mediante el Escenario 1 ───────────────────────────────────────
salida = generar_escenario_1(SIM_CFG)

# El flujo 1 analiza UNA réplica; el estudio Monte Carlo completo itera sobre R.
REPLICA_IDX = 0
X_raw = salida.observaciones[REPLICA_IDX]          # (T, L): matriz observada
T, G  = X_raw.shape

# Alias de compatibilidad: el resto del notebook usa `domain.grid`
from types import SimpleNamespace
domain = SimpleNamespace(grid=salida.grilla)

print(f"Datos simulados : {X_raw.shape}  →  T={T} curvas, G={G} puntos")
print("Control de calidad del generador:")
for k, v in salida.diagnostico.items():
    print(f"  {k:30s} = {v}")

# ── Guardar configuración de simulación y grilla ─────────────────────────────
# La grilla y las curvas se persisten juntas en el paso funcional
# mediante guardar_curvas (contrato de artefactos).

simulation_config = {
    "sim_params":    salida.config.to_dict(),
    "diagnostico":   salida.diagnostico,
    "replica_idx":   REPLICA_IDX,
    "experiment_id": EXPERIMENT_ID,
    "seed":          SEED,
    "T":             int(T),
    "G":             int(G),
}
with open(PATHS["raw"] / "simulation_config.json", "w", encoding="utf-8") as _f:
    json.dump(simulation_config, _f, indent=2, ensure_ascii=False)

# Persistencia completa del escenario (todas las réplicas, formato .npz)
_npz = guardar_escenario(salida, str(PATHS["raw"] / "escenario_1"),
                         incluir_curvas=True, incluir_internos=True)

print(f"[raw] domain_grid.npy  ({domain.grid.shape})  → {PATHS['raw']}")
print(f"[raw] simulation_config.json             → {PATHS['raw']}")
print(f"[raw] escenario_1.npz (salida completa)  → {_npz}")


## 2.2 Visualización de los datos empíricos

In [ ]:
# ── Serie de tiempo funcional empírica (línea continua desplazada) ────────────
highlight_idx = [0, 1, T // 2, T - 1]

fig = plot_fts_empirical(
    X_raw, domain.grid,
    highlight_idx   = highlight_idx,
    title           = f"FAR(1) — {T} curvas empíricas (escala original)",
    separator_every = 5,
    save_path       = str(PATHS["out_report"] / "01_fts_empirica_raw.png"),
)
plt.show()

In [ ]:
# ── Muestra de curvas empíricas (5 índices fijos) ─────────────────────────────
fig = plot_empirical_sample(
    X_raw, domain.grid,
    sample_idx = [40, 1, 2, 80, 4],
    title      = "Muestra de 5 curvas empíricas (escala original)",
    save_path  = str(PATHS["out_report"] / "02_muestra_empirica_raw.png"),
)
plt.show()

In [ ]:
# ── Media y varianza funcional (panel combinado) ──────────────────────────────
fig = plot_mean_and_variance(
    X_raw, domain.grid,
    show_std1 = True,
    show_std2 = True,
    title     = "Media y varianza funcional — FAR(1) escala original",
    save_path = str(PATHS["out_report"] / "03_media_varianza_raw.png"),
)
plt.show()

In [ ]:
# ── 5 curvas como serie de tiempo (indexadas en t) — presentación ────────────
import numpy as np, matplotlib.pyplot as plt

n_dias = 5
X   = X_raw                                    # serie simulada (T, G); el alias 'X' también sirve
g   = np.asarray(domain.grid, dtype=float)
g01 = (g - g.min()) / (g.max() - g.min())      # dominio intra-curva normalizado a [0,1]

fig, ax = plt.subplots(figsize=(12, 4.5))
for i in range(n_dias):
    ax.plot(i + g01, X[i], color="#e07b39", lw=1.6)   # curva i ocupa el intervalo [i, i+1] en t

for i in range(n_dias + 1):                             # separadores
    ax.axvline(i, color="0.75", lw=0.8, zorder=0)
for i in range(n_dias):                                 # etiqueta por curva
    ax.text(i + 0.5, 0.97, f"$X_{{{i+1}}}(\\tau)$",
            transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=10, color="#b5561f")

ax.set_xlabel(r"Tiempo $t$  (intervalo $[t-1,\,t]$ = curva $X_t(\tau)$)")
ax.set_ylabel(r"$X_t(\tau)$")
ax.set_title("Serie de tiempo funcional simulada — 5 curvas")
ax.set_xticks(range(n_dias + 1))
ax.margins(x=0.01)
fig.tight_layout()
fig.savefig(PATHS["out_report"] / "pres_5_series_t_sim.png", dpi=150, bbox_inches="tight")
plt.show()

## 2.3 Persistencia de datos crudos


In [ ]:
X = X_raw   # alias para el resto del notebook (representación funcional sobre crudos)

# ── Persistencia de curvas y grilla (contrato de artefactos) ─────────────────
# guardar_curvas escribe X_curves.npy y domain_grid.npy en 'functional', que es
# donde el flujo de resultados los busca.
_p = guardar_curvas(PATHS, X, domain.grid)
print(f"[functional] X_curves.npy    {X.shape}")
print(f"[functional] domain_grid.npy {domain.grid.shape}")

## 2.4 Partición temporal de entrenamiento y prueba

La partición respeta el orden de la serie: todo objeto **estimado a partir de
los datos** (selección GCV de la base, FPCA, estandarizador) se ajusta
exclusivamente con el bloque inicial de entrenamiento $\{1,\dots,T_0\}$ y se
aplica al bloque de prueba $\{T_0+1,\dots,T\}$ mediante `transform`. El bloque
de prueba queda reservado para la evaluación fuera de muestra.

In [ ]:
# ── Partición temporal (holdout) 
PROP_TRAIN = 0.80                          # proporción del bloque de entrenamiento
T0 = int(np.floor(PROP_TRAIN * T))         # nº de curvas de entrenamiento

assert 10 < T0 < T, f"T0={T0} fuera de rango para T={T}."

idx_train = np.arange(0, T0)               # t = 1 … T0      (0-based)
idx_test  = np.arange(T0, T)               # t = T0+1 … T
X_train, X_test = X[idx_train], X[idx_test]

print("Partición temporal (holdout):")
print(f"  entrenamiento : t ∈ [1, {T0}]      →  {X_train.shape}")
print(f"  prueba        : t ∈ [{T0+1}, {T}]  →  {X_test.shape}")
print(f"  proporción    : {T0/T:.1%} / {1 - T0/T:.1%}")

# 3. Representación funcional B-spline

In [ ]:
# SELECCIÓN DE PARÁMETROS PARA EVALUAR LA REPRESENTACIÓN FUNCIONAL (n_basis, order)
N_BASIS_RANGE = range(2, min(30, T0 // 2))   # [HOLDOUT] rango acotado por T0
ORDER_RANGE   = range(2, 5)
selection_records = []

for order_bs in ORDER_RANGE:
    for nb in N_BASIS_RANGE:
        if nb < order_bs:
            continue
        try:
            fr_tmp = FunctionalRepresentation(method="bspline", n_basis=nb, order=order_bs)
            TH_tmp = fr_tmp.fit_transform(X_train, domain.grid)   # [HOLDOUT] solo train
            X_rec  = fr_tmp.reconstruct(TH_tmp)

            L_i    = X_train.shape[1]                            # puntos por curva
            sse_c  = np.sum((X_train - X_rec) ** 2, axis=1)      # SSE_i(K)
            ss_tot = np.sum((X_train - X_train.mean(axis=0, keepdims=True)) ** 2)
            vr     = 1.0 - sse_c.sum() / ss_tot                  # VE(K)
            rmse_c = np.sqrt(sse_c / L_i)                        # e_i(K)
            df_k   = nb                                          # df(K) = n_basis
            gcv_c  = (L_i * sse_c / (L_i - df_k) ** 2            # GCV_i(K)
                      if L_i > df_k else np.full_like(sse_c, np.nan))

            selection_records.append({
                "n_basis": nb, "order": order_bs, "var_retained": vr,
                "rmse_mean": rmse_c.mean(), "rmse_max": rmse_c.max(),
                "gcv_mean": float(np.mean(gcv_c)),
            })
        except Exception as e:
            print(f"  [SKIP] n_basis={nb}, order={order_bs}: {e}")

sel_df = pd.DataFrame(selection_records)

# ── Criterio: mínimo de GCV promedio sobre las curvas ─────────────────────
sel_valid = sel_df.dropna(subset=["gcv_mean"])
best_row  = sel_valid.loc[sel_valid["gcv_mean"].idxmin()]
nb_best   = int(best_row["n_basis"])
ord_best  = int(best_row["order"])

display(sel_df.style
    .format({"var_retained": "{:.4%}", "rmse_mean": "{:.6f}",
             "rmse_max": "{:.6f}", "gcv_mean": "{:.6f}"})
    .background_gradient(subset=["gcv_mean"],     cmap="YlOrRd_r")
    .background_gradient(subset=["var_retained"], cmap="YlGn")
    .background_gradient(subset=["rmse_mean"],    cmap="YlOrRd_r"))

print(f"\nSelección por GCV mínimo: n_basis={nb_best}, order={ord_best}, "
      f"GCV={best_row['gcv_mean']:.6f}")
print(f"Verificación descriptiva: var_retained={best_row['var_retained']:.4%}, "
      f"rmse_mean={best_row['rmse_mean']:.6f}, rmse_max={best_row['rmse_max']:.6f}")


In [ ]:
# ── Heatmaps + curva de varianza retenida ────────────────────────────────────
fig = plot_seleccion_basis(
    sel_df, nb_best, ord_best,
    save_path=str(PATHS["out_report"] / "08_seleccion_basis.png"),
)
plt.show()


## 3.1 Ajuste y visualización de la representación funcional

In [ ]:
# CONSTANTES ELEGIDAS PARA LA REPRESENTACIÓN FUNCIONAL
# fr = FunctionalRepresentation(method="bspline", n_basis=nb_best, order=ord_best)
fr = FunctionalRepresentation(method="bspline", n_basis=7, order=4)


# [HOLDOUT] Ajuste SOLO con entrenamiento; proyección de toda la serie.
# Para B-splines la base depende únicamente de la grilla, pero se mantiene la
# disciplina fit/transform para que el patrón sea válido con cualquier método.
fr.fit(X_train, domain.grid)
THETA       = fr.transform(X, domain.grid)            # (T, K)  serie completa
THETA_train = THETA[idx_train]                        # (T0, K)
print(f"THETA shape: {THETA.shape}  (train={THETA_train.shape[0]}, test={T - T0})")

# ── Serie de tiempo funcional con representación B-spline ────────────────────
fig = plot_fts_functional(
    X, domain.grid,
    fr              = fr,
    highlight_idx   = highlight_idx,
    title           = f"FAR(1) — {T} curvas (repr. B-spline, n_basis={nb_best}, order={ord_best})",
    separator_every = 5,
    save_path       = str(PATHS["out_report"] / "09_fts_funcional_bspline.png"),
)
plt.show()

# ── Serializar la representación (contrato de artefactos) ────────────────────
guardar_representacion(PATHS, fr, THETA,
                       extra={"T0": int(T0), "prop_train": float(PROP_TRAIN),
                              "ajustado_en": "train"})
print(f"[functional] functional_representation.pkl + theta.csv + fr_config.json")


## 3.2 Pasar la base selecionada a FPCA analizar cuantos componentes me quedare

In [ ]:
# FPCA GENERALIZADO en métrica L² (functions_models.FPCA_L2).
# Base B-spline NO ortonormal ⇒ Gram W = <φ_j,φ_k>_{L²} ≠ I, de modo que la
# descomposición correcta resuelve el problema propio generalizado
#     (W^{1/2} S_θ W^{1/2}) z = λ z,   con ‖ψ_m‖_{L²} = 1.
# [HOLDOUT] el ajuste emplea SOLO el bloque de entrenamiento.

Phi = base_en_grilla(fr, THETA.shape[1])          # (G, K): base en grilla
fpca = FPCA_L2().fit(THETA_train, Phi, domain.grid)

# ── Verificación de las identidades que sostienen la construcción ────────────
_ver = fpca.verificar(THETA_train, fr=fr)
print("Verificación FPCA_L2 (sobre entrenamiento):")
for _k, _v in _ver.items():
    print(f"  {_k:30s} = {_v if not isinstance(_v, float) else f'{_v:.2e}'}")
assert _ver["todo_ok"], "Las identidades del FPCA generalizado no se cumplen."

# ── Diagnóstico dinámico: AR(1) propio de cada componente (≠ varianza) ───────
# Varianza = representación; AR(1) = dinámica. Una FPC de var. baja puede tener
# AR fuerte (útil para pronóstico) y una de var. alta puede ser ruido temporal.
K = fpca.evals.size
_Bf = fpca.B_full
SCORES_all = (THETA_train - fpca.mu_theta) @ (fpca.W @ _Bf)   # (T0, K)
s0, s1 = SCORES_all[:-1], SCORES_all[1:]
ar1_own = (s1 * s0).sum(0) / np.clip((s0 * s0).sum(0), 1e-12, None)

evals, var_cum = fpca.evals, fpca.var_cum
VAR_TARGET  = 0.99
M_SUGGERIDO = fpca.seleccionar_M(VAR_TARGET)

fpca_tbl = pd.DataFrame({
    "componente": np.arange(1, K + 1),
    "autovalor":  evals,
    "var_ratio":  fpca.var_ratio,
    "var_acum":   var_cum,
    "ar1_propio": ar1_own,
})
display(fpca_tbl.head(min(15, K)).style.format(
    {"autovalor": "{:.4e}", "var_ratio": "{:.4%}", "var_acum": "{:.4%}", "ar1_propio": "{:+.3f}"}
).background_gradient(subset=["var_ratio"], cmap="YlGn")
 .background_gradient(subset=["ar1_propio"], cmap="coolwarm", vmin=-1, vmax=1))
print(f"\nK B-spline disponibles : {K}")
print(f"SUGERENCIA (var ≥ {VAR_TARGET:.0%}) : M_SUGGERIDO = {M_SUGGERIDO}   "
      f"← solo referencia; fija M_FPCA en la celda 3.3")

fig = plot_fpca_scree(
    evals, var_cum, M_SUGGERIDO, var_target=VAR_TARGET,
    save_path=str(PATHS["out_report"] / "10_fpca_scree.png"),
)
plt.show()

## 3.3 Obtener el FPCA basado en la selecion

In [ ]:
M_FPCA = 5    # ← nº de componentes FPCA a retener (lo fijas TÚ)

assert isinstance(M_FPCA, (int, np.integer)) and 1 <= M_FPCA <= fpca.evals.size, (
    f"M_FPCA debe ser entero en [1, {fpca.evals.size}]. Recibido: {M_FPCA!r}"
)
fpca.set_M(int(M_FPCA))
M_fpca = fpca.M

# ── Objetos derivados (compatibilidad con el resto del notebook) ─────────────
Psi_grid = fpca.Psi_grid          # (G, M) autofunciones ortonormales en L²
mu_grid  = fpca.mu_grid           # (G,)
W        = fpca.W                 # (K, K) Gram
B        = fpca.B                 # (K, M)
mu_theta = fpca.mu_theta          # (K,)

# [HOLDOUT] scores de TODA la serie proyectados sobre la base de entrenamiento
SCORES       = fpca.transform(THETA)          # (T, M)
SCORES_train = SCORES[idx_train]
SCORES_test  = SCORES[idx_test]

# ── Chequeo fuera de muestra: deriva entre bloques (estacionariedad §2.2.1) ──
print(f"M_fpca = {M_fpca}   var. explicada = {fpca.var_cum[M_fpca-1]:.4%}")
print(f"[train] max|media ξ|  = {np.abs(SCORES_train.mean(0)).max():.2e}   (≈ 0)")
print(f"[test]  max|media ξ|  = {np.abs(SCORES_test.mean(0)).max():.3f}")
print(f"[test]  var ξ / λ     = "
      f"{np.array2string(SCORES_test.var(0, ddof=1) / fpca.lambdas, precision=3)}")

# ── Persistencia FPCA (contrato de artefactos) ───────────────────────────────
# SCORES_STD se calcula y persiste en §4.0; aquí guardamos la FPCA y los scores
# en escala original. El estandarizador añade fpca_scores_std.csv después.
guardar_fpca(PATHS, fpca, SCORES, meta_extra={"T0": int(T0)})
print(f"[functional] artefactos FPCA (eigenfunctions, mean, gram_W, coef_B, "
      f"mu_theta, basis_phi, scores, evals, meta)")

# 4. Construcción de datasets AR(p) sobre scores de FPCA (ecuación-por-ecuación)

## 4.0 Estandarización de scores ξ  **[FIX — punto de entrada al modelo]**

Los scores FPCA tienen varianza λₘ (heterogénea entre componentes:
λ₁ ≫ λ_M). Se estandarizan por columna (z-score, ddof=0) **antes** de
construir los datasets AR(p): respuesta y covariables entran al PSBP en
escala estandarizada. El estandarizador se persiste para que `02_03`
des-estandarice las predicciones antes de reconstruir curvas.

Nota: las correlaciones de rezagos (§4.1) son invariantes a esta
transformación lineal; se calculan sobre `SCORES_STD` por consistencia.

In [ ]:
scores_standardizer = DataStandardizer(method="zscore_column", ddof=0)
scores_standardizer.fit(SCORES_train)                 # [HOLDOUT] fit SOLO en train
SCORES_STD       = scores_standardizer.transform(SCORES)   # aplicado a toda la serie
SCORES_STD_train = SCORES_STD[idx_train]
SCORES_STD_test  = SCORES_STD[idx_test]

print(scores_standardizer.summary())
print(f"SCORES_STD : shape={SCORES_STD.shape}  (train={T0}, test={T - T0})")
print(f"  [train] max|media| = {np.abs(SCORES_STD_train.mean(0)).max():.2e}  (≈ 0 por construcción)")
print(f"  [train] max|std-1| = {np.abs(SCORES_STD_train.std(0) - 1).max():.2e}  (≈ 0 por construcción)")
print(f"  [test]  media      = {np.array2string(SCORES_STD_test.mean(0), precision=3)}")
print(f"  [test]  std        = {np.array2string(SCORES_STD_test.std(0),  precision=3)}")
print(f"  std train por componente (=√λ_m): "
      f"{np.array2string(SCORES_train.std(axis=0), precision=3)}")

# ── Persistencia (contrato de artefactos) ────────────────────────────────────
guardar_estandarizador(PATHS, scores_standardizer)
# Re-persistir la FPCA incluyendo ahora los scores estandarizados
guardar_fpca(PATHS, fpca, SCORES, SCORES_STD=SCORES_STD, meta_extra={"T0": int(T0)})
print(f"[functional] scores_standardizer/ + fpca_scores_std.csv")

# ── Diagnóstico visual (media/std por componente, antes y después) ───────────
fig = plot_diagnostico_estandarizacion(
    SCORES_train, SCORES_STD_train, np.arange(1, M_fpca + 1),
    labels=("Scores ξ (escala λ)", "Scores ξ estandarizados"),
    title="estadísticas por componente FPCA",
    save_path=str(PATHS["out_report"] / "04_diagnostico_estandarizacion_scores.png"),
)
fig.axes[0].set_xlabel("componente m"); fig.axes[1].set_xlabel("componente m")
plt.show()


In [ ]:
# [HOLDOUT] la elección de rezagos es una decisión de modelado: solo train
T_theta = SCORES_STD_train.shape[0]
K_total = SCORES_STD_train.shape[1]                          # nº de FPC disponibles (M)

# Lags de Diagnostico
N_LAGS_MAX = 3
N_LAGS_MAX = int(np.clip(N_LAGS_MAX, 1, T_theta - 2))

def _spearman_block(Y, X):
    """Spearman columna-a-columna vía rangos (pandas; sin scipy)."""
    Yr = pd.DataFrame(Y).rank().to_numpy()
    Xr = pd.DataFrame(X).rank().to_numpy()
    Yc = Yr - Yr.mean(0); Xc = Xr - Xr.mean(0)
    return (Yc.T @ Xc) / np.outer(np.sqrt((Yc**2).sum(0)), np.sqrt((Xc**2).sum(0)))

# ── Matrices de correlación: respuesta (t) × predictor (FPC j, lag ℓ) ────────
n_cov = K_total * N_LAGS_MAX
corr_pearson  = np.zeros((K_total, n_cov))
corr_spearman = np.zeros((K_total, n_cov))
col_labels = []
y_block = SCORES_STD_train[N_LAGS_MAX:, :]                         # respuesta en t (alineada)

for lag in range(1, N_LAGS_MAX + 1):
    x_block = SCORES_STD_train[N_LAGS_MAX - lag : T_theta - lag, :]   # predictores en t-lag
    sp = _spearman_block(y_block, x_block)
    for j in range(K_total):
        c = (lag - 1) * K_total + j
        for k in range(K_total):
            corr_pearson[k, c] = np.corrcoef(y_block[:, k], x_block[:, j])[0, 1]
        corr_spearman[:, c] = sp[:, j]
        col_labels.append(rf"$\xi_{{t-{lag},{j+1}}}$")

row_labels = [rf"$\xi_{{t,{k+1}}}$" for k in range(K_total)]
band  = 1.96 / np.sqrt(len(y_block))                     # banda de significancia (referencia)
VCLIP = 0.6

# ── Heatmaps de correlación ───────────────────────────────────────────────────
fig = plot_rezagos_heatmap(
    corr_pearson, col_labels, row_labels,
    title=f"Pearson — respuesta(t) vs lags 1..{N_LAGS_MAX}",
    n_lags_max=N_LAGS_MAX, K_total=K_total, band=band, vclip=VCLIP,
    save_path=str(PATHS["out_report"] / "12a_rezagos_pearson.png"),
)
plt.show()

fig = plot_rezagos_heatmap(
    corr_spearman, col_labels, row_labels,
    title=f"Spearman — respuesta(t) vs lags 1..{N_LAGS_MAX}",
    n_lags_max=N_LAGS_MAX, K_total=K_total, band=band, vclip=VCLIP,
    save_path=str(PATHS["out_report"] / "12b_rezagos_spearman.png"),
)
plt.show()


## 4.1 Selecion de lags y DATSET final 

In [ ]:
N_LAGS  = 3
K_total = SCORES_STD.shape[1]        # nº de FPC disponibles (M)
COMPONENT_IDX = list(range(K_total))

# [HOLDOUT] tamaños efectivos por bloque
n_train_eff = T0 - N_LAGS            # respuestas en t = N_LAGS+1 … T0
n_test_eff  = T - T0                 # respuestas en t = T0+1 … T (rezagos desde train)

assert T0 > N_LAGS, f"T0={T0} debe superar N_LAGS={N_LAGS}."

print(f"K_total disponibles : {K_total}  (índices 0 … {K_total - 1})")
print(f"T / T0              : {T} / {T0}")
print(f"N_LAGS              : {N_LAGS}")
print(f"n_train_eff         : {n_train_eff}")
print(f"n_test_eff          : {n_test_eff}")

In [ ]:
# ── Validación ───────────────────────────────────────────────────────────────
assert len(COMPONENT_IDX) > 0, "COMPONENT_IDX no puede estar vacío."
assert len(COMPONENT_IDX) == len(set(COMPONENT_IDX)), "COMPONENT_IDX tiene índices repetidos."
assert all(0 <= i < K_total for i in COMPONENT_IDX), (
    f"Todos los índices deben estar en [0, {K_total - 1}]. Recibido: {COMPONENT_IDX}"
)

n_components = len(COMPONENT_IDX)

# ── Resumen ───────────────────────────────────────────────────────────────────
print(f"K_total disponibles : {K_total}")
print(f"Componentes usados  : {n_components}  →  índices {COMPONENT_IDX}")
print(f"Orden AR (N_LAGS)   : {N_LAGS}")
print()
print(f"  {'k_modelo':>8}  {'idx_THETA':>10}  {'nombre_resp':>14}")
print(f"  {'─'*8}  {'─'*10}  {'─'*14}")
for k_model, idx in enumerate(COMPONENT_IDX):
    print(f"  {k_model:>8}  {idx:>10}  {'fpc_' + str(idx + 1):>14}")

In [ ]:
# ── Construcción de DataFrames AR(p): bloques de entrenamiento y prueba ──────
SCORES_sel = SCORES_STD[:, COMPONENT_IDX]   # (T, n_components) estandarizados

cov_names = [
    f"fpc_{COMPONENT_IDX[j] + 1}_lag{lag}"
    for lag in range(1, N_LAGS + 1)
    for j in range(n_components)
]

def _dataset_bloque(k: int, t_ini: int, t_fin: int) -> pd.DataFrame:
    """
    Filas con respuesta en t ∈ [t_ini, t_fin) y predictores en t-1 … t-N_LAGS.

    Los rezagos del primer origen de prueba provienen del final del bloque de
    entrenamiento: son observaciones pasadas disponibles en cada origen, de
    modo que su uso es el condicionamiento que prescribe §2.2.3.1 (predicción
    a horizonte h=1 con la historia observada), no fuga de información.
    """
    t_idx  = np.arange(t_ini, t_fin)
    y_col  = SCORES_sel[t_idx, k]
    X_cols = np.hstack([SCORES_sel[t_idx - lag, :] for lag in range(1, N_LAGS + 1)])
    return pd.DataFrame(np.column_stack([y_col, X_cols]),
                        columns=[f"fpc_{COMPONENT_IDX[k] + 1}"] + cov_names)

dfs_train, dfs_test = {}, {}
for k in range(n_components):
    dfs_train[k] = _dataset_bloque(k, N_LAGS, T0)     # (T0 - N_LAGS) filas
    dfs_test[k]  = _dataset_bloque(k, T0,     T)      # (T - T0) filas

dfs = dfs_train   # alias: el bloque de hiperparámetros itera sobre dfs

# ── Manifest + persistencia (contrato de artefactos) ─────────────────────────
manifest = {
    "scores_scale":  "standardized_zscore_ddof0",
    "n_components":  n_components,
    "n_lags":        int(N_LAGS),
    "component_idx": [int(i) for i in COMPONENT_IDX],
    "cov_names":     cov_names,
    "T":             int(T),
    "T0":            int(T0),
    "prop_train":    float(PROP_TRAIN),
    "n_train_eff":   int(n_train_eff),
    "n_test_eff":    int(n_test_eff),
    "ajuste_en":     "train",
}
guardar_datasets_ar(PATHS, dfs_train, dfs_test, manifest)
print(f"[functional] {2*n_components} datasets (train/test) + datasets_manifest.json")

# 5. Especificación de hiperparámetros y ajuste MCMC

In [ ]:
MCMC_CONFIG = {"nsim": 2000, "burn": 500, "N": 20, "M": 50}
N_CHAINS    = 3
BURN        = int(MCMC_CONFIG["burn"])
print(f"MCMC_CONFIG : {MCMC_CONFIG}")
print(f"N_CHAINS    : {N_CHAINS}  (réplicas que ejecutará MATLAB)")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# HIPERPARÁMETROS — priors heterogéneas por tipo de variable
# ════════════════════════════════════════════════════════════════════════════

# ── Escalares globales ────────────────────────────────────────────────────────
HP_GLOBAL = {
    "atau":  2.0,
    "btau":  0.5,
    "ag":    2.0,
    "bg":    0.5,
    "mumu":  0.0,
    "taumu": 1.0,
    "pwj":   0.5,
}

# ── Priors por tipo de variable ───────────────────────────────────────────────
#   Formato: "tipo": (apij, bpij, mupsij, taupsij)
HP_BY_TYPE = {
    "own_lag1":  (9.0, 1.0,  0.0, 1.0),   # E[π] = 0.90
    "cross_lag": (1.0, 1.0,  0.0, 1.0),   # E[π] = 0.50
}

# ── Clasificador ──────────────────────────────────────────────────────────────
def _classify(name: str, k_model: int, component_idx: list) -> str:
    own_name = f"fpc_{component_idx[k_model] + 1}_lag1"
    if name == own_name:
        return "own_lag1"
    if "_lag" in name:
        return "cross_lag"
    return "cross_lag"   # fallback: cualquier variable no reconocida → débil

# ── Construcción automática de HYPERPARAMS_LIST ───────────────────────────────
HYPERPARAMS_LIST = []
for k in range(n_components):
    p = len(cov_names)
    apij    = np.empty(p); bpij    = np.empty(p)
    mupsij  = np.empty(p); taupsij = np.empty(p)

    for j, name in enumerate(cov_names):
        vtype              = _classify(name, k, COMPONENT_IDX)
        a, b, mu, tau      = HP_BY_TYPE[vtype]
        apij[j]    = a;    bpij[j]    = b
        mupsij[j]  = mu;   taupsij[j] = tau

    HYPERPARAMS_LIST.append({**HP_GLOBAL,
                              "apij": apij,     "bpij": bpij,
                              "mupsij": mupsij, "taupsij": taupsij})

# ── Tabla de resumen ──────────────────────────────────────────────────────────
_W = 70
print("═" * _W)
print(f"  HYPERPARAMS_LIST  —  {n_components} componentes × {p} variables")
print("═" * _W)
print(f"  Globales: atau={HP_GLOBAL['atau']} btau={HP_GLOBAL['btau']}  "
      f"ag={HP_GLOBAL['ag']} bg={HP_GLOBAL['bg']}  "
      f"mumu={HP_GLOBAL['mumu']} taumu={HP_GLOBAL['taumu']}  "
      f"pwj={HP_GLOBAL['pwj']}")
print()
for k in range(n_components):
    hp = HYPERPARAMS_LIST[k]
    print(f"  Componente k={k+1}  (fpc_{COMPONENT_IDX[k]+1})")
    print(f"  {'Variable':<26} {'Tipo':<12} {'apij':>6} {'bpij':>6} "
          f"{'E[π]':>6} {'mupsij':>8} {'taupsij':>9}")
    print(f"  {'─'*26} {'─'*12} {'─'*6} {'─'*6} "
          f"{'─'*6} {'─'*8} {'─'*9}")
    for j, name in enumerate(cov_names):
        vtype = _classify(name, k, COMPONENT_IDX)
        a  = hp["apij"][j];    b  = hp["bpij"][j]
        mu = hp["mupsij"][j];  tau = hp["taupsij"][j]
        e_pi = a / (a + b)
        marker = "  ◄" if vtype == "own_lag1" else ""
        print(f"  {name:<26} {vtype:<12} {a:>6.1f} {b:>6.1f} "
              f"{e_pi:>6.3f} {mu:>8.1f} {tau:>9.1f}{marker}")
    print()

In [ ]:
# ── Guardar hiperparámetros del modelo en artefact ───────────────────────────

hp_artifact = {
    "global": HP_GLOBAL,
    "by_type": HP_BY_TYPE,
    "mcmc_config": MCMC_CONFIG,
    "n_iter": N_CHAINS,
    # [FIX] esquema real de semillas usado por psbp_fd_iteracion.m
    "seed_scheme": "SEED_BASE + tt*9973 + k*31",
    "seed_base": SEED,
    "scores_scale": "standardized_zscore_ddof0",
    # [HOLDOUT] partición temporal — MATLAB entrena SOLO con *_train.csv
    "partition": {
        "T": int(T), "T0": int(T0), "prop_train": float(PROP_TRAIN),
        "n_train_eff": int(n_train_eff), "n_test_eff": int(n_test_eff),
        "train_files": [f"dataset_fpc_{COMPONENT_IDX[k]+1}_train.csv"
                        for k in range(n_components)],
        "test_files":  [f"dataset_fpc_{COMPONENT_IDX[k]+1}_test.csv"
                        for k in range(n_components)],
    },
    "hyperparams_list": []
}

for k in range(n_components):
    hp = HYPERPARAMS_LIST[k].copy()
    hp_artifact["hyperparams_list"].append({
        "component_k": k,
        "fpc_idx": int(COMPONENT_IDX[k] + 1),
        "hyperparams": {
            key: value.tolist() if isinstance(value, np.ndarray) else value
            for key, value in hp.items()
        }
    })

guardar_hiperparametros(PATHS, hp_artifact)

print(f"✓ Hiperparámetros guardados en: {PATHS['out_artefact'] / 'hyperparameters.json'}")

# 6. Configuración de evaluación y líneas base (bloque de prueba)

Se persiste la configuración de evaluación (§2.2.3) que consumirá el flujo de
resultados, y se calculan **líneas base** sobre el bloque de prueba: el
predictor ingenuo de persistencia ($\hat\xi_t = \xi_{t-1}$) y la media
incondicional de entrenamiento ($\hat\xi_t = 0$ en escala estandarizada).
Ambas se evalúan a nivel de scores (RMSE) y a nivel de curva (MISE, RMSE
funcional), y constituyen el piso que todo modelo debe superar para justificar
su complejidad. No requieren MCMC, por lo que viven en este flujo.

In [ ]:
# ── Configuración de evaluación (la consume el flujo de resultados) ──────────
eval_config = {
    "scheme":        "holdout_temporal",       # §2.2.3.1
    "T":             int(T),
    "T0":            int(T0),
    "prop_train":    float(PROP_TRAIN),
    "horizons":      [1],                       # h=1 vía dataset_test; h>1: simulación
    "n_lags":        int(N_LAGS),
    "metrics_scores": ["RMSE", "R2"],           # §2.2.3.2 sobre coeficientes
    "metrics_curvas": ["MISE", "RMSE_funcional"],
    "metrics_dist":   ["LPS", "CRPS", "cobertura_95", "PIT"],   # §2.2.3.3 (flujo 3)
    "scores_scale":  "standardized_zscore_ddof0",
}
guardar_config_evaluacion(PATHS, eval_config)
print(f"[out_artefact] eval_config.json")

# ── Líneas base sobre el bloque de prueba (h=1) ──────────────────────────────
# media incondicional (ξ̂=0) y persistencia (ξ̂_t = ξ_{t-1}), evaluadas a nivel
# de scores (RMSE, R²) y de curva (MISE, RMSE funcional) con fit.tabla_baselines.
baselines_df = tabla_baselines(
    SCORES_STD, T0,
    estandarizador=scores_standardizer,
    fpca=fpca,
    X_obs=X,
    tau=domain.grid,
    h=1,
)
baselines_df.to_csv(PATHS["out_report"] / "32_baselines_test.csv")
print(f"[out_report] 32_baselines_test.csv")
display(baselines_df.style.format("{:.4f}", na_rep="—")
        .set_caption("Líneas base — bloque de prueba, h=1 "
                     "(piso que el PSBP-FD debe superar)"))

# ── Verificación cruzada del contrato de artefactos ──────────────────────────
# Cruza manifest ↔ hyperparameters ↔ FPCA antes de pasar a MATLAB / flujo 3.
informe = verificar_contrato(PATHS)
print(f"\ncontrato_ok = {informe['contrato_ok']}  "
      f"(M={informe['M']}, K={informe['K']}, T0={informe['T0']}, "
      f"n_components={informe['n_components']})")